# Silver Layer — ICU chartevents

## READINESS: partly profiled, rules drafted, awaiting remaining profiling and team approval

The key, missingness, timestamp and relationship counts recorded in **Profiling Findings**
come from the run of 2026-10-19 (saved outputs of the previous version of this notebook).
Checks marked *not yet profiled* must be run and recorded before approval. Until the
`profiling_approved` widget is set to `true`, the publish step raises an error and nothing
is written. The dependency-ordered runner excludes this notebook by default.

## Source
`<source_schema>.chartevents` (default `cdr_mimiciv.bronze.chartevents`)

## Target
`<target_schema>.chartevents` (default `cdr_mimiciv.silver.chartevents`), published with
CREATE OR REPLACE semantics. Downstream Gold (`fact_icu_observation`) depends on it.
No tables are dropped.

## Grain (official MIMIC-IV icu documentation, `chartevents`)
One row per charted observation for a patient during an ICU stay (source: MetaVision).
The documentation defines **no unique row identifier**. Documented fields:
`subject_id`, `hadm_id`, `stay_id` (patient, hospital stay and ward stay identifiers;
links to patients, admissions and icustays), `caregiver_id` (the caregiver who documented
the data), `charttime` (time the observation was made; usually the closest proxy to when it
was measured), `storetime` (time the observation was manually input or manually validated
by clinical staff), `itemid` (measurement type; links to `d_items`), `value` (the value
measured), `valuenum` (the same value in numeric form if numeric, otherwise NULL; for
scores such as GCS, RASS and Code Status `valuenum` holds the score and `value` holds the
score and its text), `valueuom` (unit of measurement, if appropriate) and `warning`
(whether a warning was manually documented by the care provider).
The documentation does not state NOT NULL constraints for any column.

[DOC] Laboratory values are frequently copied into chartevents for display on the chart;
where chartevents and labevents disagree, **labevents is the ground truth**. Silver does
not reconcile the two tables: choosing the lab source is an analytical decision for Gold.

[DOC] The uploaded documentation (v2.2) lists 313,645,063 rows; Bronze holds
432,997,491, consistent with a later MIMIC-IV release. The count difference is expected.

## Silver transformation (derived from the profiling findings)
Principle, as in admissions and labevents: preserve source value, identify anomaly, add a
derived column or DQ flag, let downstream users decide.
- Select every Bronze column unchanged, in its Bronze type and order.
- Add DQ flags (our engineering choice), both non-null booleans:
  - `dq_missing_observation`: `value` and `valuenum` both NULL (1 row profiled).
  - `dq_storetime_before_charttime`: `storetime < charttime` (36,530,969 rows profiled).
    Informational, not asserted to be an error: the documentation does not say either
    order is impossible, and profiled examples show values charted at a round time
    (e.g. 15:00) and stored minutes earlier. Same flag name and meaning as outputevents
    and labevents.
- Add `_silver_processed_at`.
- No deduplication, no NULL filling, no timestamp correction, no unit conversion,
  no row removal. `value` and `valuenum` are never overwritten.
- Rows are not filtered to the ICU stay window and labevents copies are not removed:
  both are analytical choices for Gold.

## Open profiling questions
1. Are there exact duplicate rows? (No documented key; checked only when
   `run_duplicate_check` is `true` because it shuffles the full table.)
2. Missingness of every column, in particular `caregiver_id`, `storetime`, `value`,
   `valueuom`, `warning`.
3. Rows where `value` looks numeric but `valuenum` is NULL, or `valuenum` is present but
   `value` is NULL.
4. Values of `warning` and `valueuom` (case, whitespace or empty-string variants).
5. Timestamp order: size of the `storetime` before `charttime` gap; NULL `charttime`.
6. Relationships (observational only): `(stay_id, subject_id, hadm_id)` in icustays;
   `itemid` in `d_items` with `linksto = 'chartevents'`; `caregiver_id` in caregiver;
   `charttime` outside the ICU stay `intime`/`outtime` window.

## Cost note
`chartevents` is the largest table (about 433 million rows). Profiling uses a single
aggregate pass, relationship checks take distinct keys or broadcast small reference tables,
the exact-duplicate check is off by default, no `exceptAll` is used, and the published table
is checked with one aggregate pass.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("run_duplicate_check", "false")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
RUN_DUPLICATE_CHECK = dbutils.widgets.get("run_duplicate_check").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.chartevents"
TARGET_TABLE = f"{SILVER}.chartevents"

DOCUMENTED_COLUMNS = [
    "subject_id", "hadm_id", "stay_id", "caregiver_id", "charttime", "storetime",
    "itemid", "value", "valuenum", "valueuom", "warning"
]

In [0]:
chartevents_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = chartevents_bronze.columns

chartevents_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
print("Target exists:", spark.catalog.tableExists(TARGET_TABLE))

# Delta metadata without scanning rows
display(spark.sql(f"DESCRIBE DETAIL {SOURCE_TABLE}").select("format", "numFiles", "sizeInBytes", "partitionColumns"))
display(chartevents_bronze.limit(10))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- valuenum: double (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- warning: integer (nullable = true)

Documented columns missing from Bronze: []
Bronze columns not in documentation: []
Target exists: True


format,numFiles,sizeInBytes,partitionColumns
delta,1,1377159703,List()


subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
10000032,29079034,39553978,18704,2180-07-23T12:36:00.000Z,2180-07-23T14:45:00.000Z,226512,39.4,39.4,kg,0
10000032,29079034,39553978,18704,2180-07-23T12:36:00.000Z,2180-07-23T14:45:00.000Z,226707,60,60.0,Inch,0
10000032,29079034,39553978,18704,2180-07-23T12:36:00.000Z,2180-07-23T14:45:00.000Z,226730,152,152.0,cm,0
10000032,29079034,39553978,18704,2180-07-23T14:00:00.000Z,2180-07-23T14:18:00.000Z,220048,SR (Sinus Rhythm),null,null,0
10000032,29079034,39553978,18704,2180-07-23T14:00:00.000Z,2180-07-23T14:18:00.000Z,224642,Oral,null,null,0
10000032,29079034,39553978,18704,2180-07-23T14:00:00.000Z,2180-07-23T14:18:00.000Z,224650,None,null,null,0
10000032,29079034,39553978,18704,2180-07-23T14:00:00.000Z,2180-07-23T14:20:00.000Z,223761,98.7,98.7,°F,0
10000032,29079034,39553978,18704,2180-07-23T14:11:00.000Z,2180-07-23T14:17:00.000Z,220179,84,84.0,mmHg,0
10000032,29079034,39553978,18704,2180-07-23T14:11:00.000Z,2180-07-23T14:17:00.000Z,220180,48,48.0,mmHg,0
10000032,29079034,39553978,18704,2180-07-23T14:11:00.000Z,2180-07-23T14:17:00.000Z,220181,56,56.0,mmHg,0


In [0]:
# single-pass profile: missingness of every column, value consistency and timestamp order
# (no repeated scans, no shuffles)
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

NUMERIC_TEXT = r"^\s*-?[0-9]+(\.[0-9]+)?\s*$"
gap_seconds = F.unix_timestamp("charttime") - F.unix_timestamp("storetime")

profile_exprs = [F.count("*").alias("total_rows")]
profile_exprs += [_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS]
profile_exprs += [
    _cnt(F.col("value").isNull() & F.col("valuenum").isNull()).alias("value_and_valuenum_null"),
    _cnt(F.col("valuenum").isNull() & F.col("value").isNotNull()).alias("valuenum_null_value_present"),
    _cnt(F.col("valuenum").isNull() & F.col("value").rlike(NUMERIC_TEXT)).alias("valuenum_null_value_numeric_text"),
    _cnt(F.col("valuenum").isNotNull() & F.col("value").isNull()).alias("valuenum_present_value_null"),
    _cnt(F.trim(F.col("valueuom")) == "").alias("valueuom_empty_string"),
    _cnt(F.col("storetime") < F.col("charttime")).alias("storetime_before_charttime"),
    _cnt(gap_seconds > 3600).alias("storetime_before_charttime_over_1h"),
    _cnt(gap_seconds > 86400).alias("storetime_before_charttime_over_24h"),
    _cnt(F.col("storetime") == F.col("charttime")).alias("storetime_equals_charttime"),
]

chart_profile = chartevents_bronze.agg(*profile_exprs).first().asDict()
for k, v in chart_profile.items():
    print(f"{k}: {v}")

total_rows: 432997491
null_subject_id: 0
null_hadm_id: 0
null_stay_id: 0
null_caregiver_id: 15714257
null_charttime: 0
null_storetime: 583305
null_itemid: 0
null_value: 1
null_valuenum: 263786245
null_valueuom: 327131226
value_and_valuenum_null: 1
valuenum_null_value_present: 263786244
valuenum_null_value_numeric_text: 439110
valuenum_present_value_null: 0
valueuom_empty_string: 0
storetime_before_charttime: 36530969
storetime_before_charttime_over_1h: 1427616
storetime_before_charttime_over_24h: 7710
storetime_equals_charttime: 51808843


In [0]:
# exact duplicate rows: no documented key, so the only grain check is on all columns.
# Full shuffle of ~433M rows: runs only when run_duplicate_check=true.
if RUN_DUPLICATE_CHECK:
    duplicate_groups = (
        chartevents_bronze.groupBy(*SOURCE_COLUMNS).count().filter(F.col("count") > 1)
    )
    dup_summary = duplicate_groups.agg(
        F.count("*").alias("duplicate_groups"),
        F.sum(F.col("count") - 1).alias("surplus_rows"),
    ).first().asDict()
    print("Exact duplicates:", dup_summary)
    if dup_summary["duplicate_groups"]:
        display(duplicate_groups.orderBy(F.desc("count")).limit(50))
else:
    print("Exact-duplicate check skipped (run_duplicate_check=false)")

Exact-duplicate check skipped (run_duplicate_check=false)


In [0]:
# categorical values (small result sets)
print("=== warning ===")
display(chartevents_bronze.groupBy("warning").count().orderBy(F.desc("count")))

print("=== valueuom (top 100) ===")
display(chartevents_bronze.groupBy("valueuom").count().orderBy(F.desc("count")).limit(100))

=== warning ===


warning,count
0,423172272
1,9241914
null,583305


=== valueuom (top 100) ===


valueuom,count
null,327131226
mmHg,34642251
insp/min,12999774
%,12624600
bpm,10582283
cmH2O,4627027
mEq/L,4252863
mg/dL,3908524
L/min,3872067
mL,3785690


In [0]:
# text values present where valuenum is NULL (expected: categorical charting such as rhythm, route)
display(
    chartevents_bronze
    .where(F.col("valuenum").isNull())
    .groupBy("itemid", "value")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)
)

itemid,value,count
224650,None,5833784
220048,SR (Sinus Rhythm),4653541
224080,30 Degrees,2233997
224082,Side to Side,1907800
224086,Tolerated Well,1898116
224089,Compression sleeve,1820779
224642,Oral,1707013
227969,Adequate lighting,1687308
225054,On,1593235
227969,Bed locked in low position,1544869


In [0]:
# earlier profiling questions: the missing observation and examples of storetime before charttime
display(
    chartevents_bronze
    .where(F.col("value").isNull() & F.col("valuenum").isNull())
    .limit(20)
)

display(
    chartevents_bronze
    .where(F.col("storetime") < F.col("charttime"))
    .select("stay_id", "itemid", "charttime", "storetime", gap_seconds.alias("seconds_early"))
    .limit(20)
)

subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
14998916,22487973,37386043,73481,2176-03-30T19:29:00.000Z,2176-03-30T19:29:00.000Z,226748,null,null,null,0


stay_id,itemid,charttime,storetime,seconds_early
39553978,224080,2180-07-23T15:00:00.000Z,2180-07-23T14:32:00.000Z,1680
39553978,224082,2180-07-23T15:00:00.000Z,2180-07-23T14:32:00.000Z,1680
39553978,224084,2180-07-23T15:00:00.000Z,2180-07-23T14:32:00.000Z,1680
39553978,224086,2180-07-23T15:00:00.000Z,2180-07-23T14:32:00.000Z,1680
39553978,224093,2180-07-23T15:00:00.000Z,2180-07-23T14:32:00.000Z,1680
39553978,228868,2180-07-23T15:00:00.000Z,2180-07-23T14:32:00.000Z,1680
39553978,223761,2180-07-23T20:00:00.000Z,2180-07-23T19:59:00.000Z,60
39553978,224642,2180-07-23T20:00:00.000Z,2180-07-23T19:59:00.000Z,60
37081114,224168,2150-11-03T12:00:00.000Z,2150-11-03T11:55:00.000Z,300
37081114,224641,2150-11-03T12:00:00.000Z,2150-11-03T11:55:00.000Z,300


In [0]:
# relationships (observational only; findings are recorded, not turned into flags here)
# reference tables are small and broadcast; the large table is scanned once for these checks
icustays_table = f"{SILVER}.icustays"
d_items_table = f"{SILVER}.d_items"
caregiver_table = f"{SILVER}.caregiver"

for _t in (icustays_table, d_items_table, caregiver_table):
    assert spark.catalog.tableExists(_t), f"Missing reference: {_t}"

icu_ref = (
    spark.table(icustays_table)
    .select("stay_id", "subject_id", "hadm_id", "intime", "outtime",
            F.lit(True).alias("_stay_match"))
)
item_ref = spark.table(d_items_table).select("itemid", F.col("linksto").alias("_linksto"))
# caregiver_id is a string in Silver caregiver; cast to INT for the comparison
caregiver_ref = (
    spark.table(caregiver_table)
    .select(F.expr("TRY_CAST(trim(caregiver_id) AS INT)").alias("caregiver_id"))
    .where(F.col("caregiver_id").isNotNull())
    .distinct()
    .withColumn("_caregiver_match", F.lit(True))
)

rel = (
    chartevents_bronze.select("stay_id", "subject_id", "hadm_id", "caregiver_id", "itemid", "charttime")
    .join(F.broadcast(icu_ref), ["stay_id", "subject_id", "hadm_id"], "left")
    .join(F.broadcast(item_ref), "itemid", "left")
    .join(F.broadcast(caregiver_ref), "caregiver_id", "left")
    .agg(
        _cnt(F.col("_stay_match").isNull()).alias("rows_stay_not_in_icustays"),
        _cnt(F.col("_linksto").isNull()).alias("rows_itemid_not_in_d_items"),
        _cnt(F.col("_linksto").isNotNull() & (F.col("_linksto") != "chartevents")).alias("rows_itemid_linksto_other_table"),
        _cnt(F.col("caregiver_id").isNotNull() & F.col("_caregiver_match").isNull()).alias("rows_caregiver_not_in_caregiver"),
        _cnt(F.col("charttime") < F.col("intime")).alias("rows_charttime_before_intime"),
        _cnt(F.col("charttime") > F.col("outtime")).alias("rows_charttime_after_outtime"),
    )
    .first().asDict()
)
for k, v in rel.items():
    print(f"{k}: {v}")

rows_stay_not_in_icustays: 0
rows_itemid_not_in_d_items: 0
rows_itemid_linksto_other_table: 0
rows_caregiver_not_in_caregiver: 0
rows_charttime_before_intime: 2322565
rows_charttime_after_outtime: 407966


## Profiling Findings

Recorded from the profiling run of 2026-10-19 on `cdr_mimiciv.bronze.chartevents`
(saved outputs of the previous notebook version). Rows marked *not yet profiled* must be
filled from the cells above before `profiling_approved` is set.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows | 432,997,491 | Row count preserved; [DOC] v2.2 doc lists 313,645,063 (earlier release) |
| Unique row key | None documented | No key asserted; no deduplication |
| Exact duplicate rows | *not yet profiled* | Report only; never removed [OURS] |
| NULL `subject_id`, `hadm_id`, `stay_id`, `itemid`, `charttime` | 0 | None |
| NULL `caregiver_id`, `storetime`, `value`, `valueuom`, `warning` | *not yet profiled* | Keep |
| NULL `valuenum` | 263,786,245 (60.9%) | Keep; [DOC] NULL when the value is not numeric |
| `value` and `valuenum` both NULL | 1 (stay 37386043, itemid 226748, 2176-03-30 19:29) | `dq_missing_observation` [OURS]; row kept |
| `valuenum` NULL but `value` numeric text | *not yet profiled* | Decide after profiling; `valuenum` never overwritten |
| `valuenum` present but `value` NULL | *not yet profiled* | Decide after profiling |
| `warning` values | *not yet profiled* | No change expected |
| `valueuom` values / empty strings | *not yet profiled* | No `valueuom_std` unless profiling shows variants [OURS] |
| `storetime` before `charttime` | 36,530,969 (8.4%); examples 1 to 28 min before a round charttime | `dq_storetime_before_charttime` [OURS]; informational, no correction |
| Gap over 1 h / over 24 h | *not yet profiled* | Informational |
| `(stay_id, subject_id, hadm_id)` not in icustays | 0 distinct | None |
| `itemid` not in `d_items` | 0 distinct | None |
| `itemid` with `linksto` other than chartevents | *not yet profiled* | Decide after profiling |
| `caregiver_id` not in caregiver | 0 distinct | None |
| `charttime` outside ICU `intime`/`outtime` | *not yet profiled* | Report only; window filtering is a Gold choice [OURS] |
| Lab values copied from labevents | Not checked in Silver | [DOC] labevents is ground truth; reconciliation is a Gold choice |

In [0]:
# Silver transformation: all source columns unchanged + DQ flags
# (our engineering choices, see Profiling Findings); no rows removed, no source value changed
chartevents_silver = chartevents_bronze.select(
    *SOURCE_COLUMNS,
    (F.col("value").isNull() & F.col("valuenum").isNull())
        .alias("dq_missing_observation"),
    F.coalesce(F.col("storetime") < F.col("charttime"), F.lit(False))
        .alias("dq_storetime_before_charttime"),
    F.current_timestamp().alias("_silver_processed_at"),
)

expected_schema = {f.name: f.dataType.simpleString() for f in chartevents_bronze.schema.fields}
expected_schema.update({
    "dq_missing_observation": "boolean",
    "dq_storetime_before_charttime": "boolean",
    "_silver_processed_at": "timestamp",
})

# profiled expected counts (2026-10-19 run)
PROFILED_COUNTS = {
    "silver_rows": 432997491,
    "dq_missing_observation": 1,
    "dq_storetime_before_charttime": 36530969,
}

chartevents_silver.printSchema()

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- valuenum: double (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- warning: integer (nullable = true)
 |-- dq_missing_observation: boolean (nullable = false)
 |-- dq_storetime_before_charttime: boolean (nullable = false)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "chartevents is not approved for publication: run and review the profiling cells, "
        "fill the 'not yet profiled' findings and any justified rules, then set widget "
        "profiling_approved=true."
    )

# cheap pre-publish validation (no data scan)
pre_schema = {f.name: f.dataType.simpleString() for f in chartevents_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    chartevents_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.chartevents


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze
# (no exceptAll on a table of this size)
published = spark.table(TARGET_TABLE)

post = published.agg(
    F.count("*").alias("silver_rows"),
    _cnt(F.col("_silver_processed_at").isNull()).alias("null_processed_at"),
    _cnt(F.col("dq_missing_observation").isNull() | F.col("dq_storetime_before_charttime").isNull())
        .alias("null_dq_flags"),
    _cnt(F.col("dq_missing_observation")).alias("dq_missing_observation"),
    _cnt(F.col("dq_storetime_before_charttime")).alias("dq_storetime_before_charttime"),
    _cnt(
        (F.col("dq_missing_observation") != (F.col("value").isNull() & F.col("valuenum").isNull()))
        | (F.col("dq_storetime_before_charttime")
           != F.coalesce(F.col("storetime") < F.col("charttime"), F.lit(False)))
    ).alias("flag_mismatches"),
).first().asDict()
bronze_rows = chartevents_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
assert post["null_processed_at"] == 0
assert post["null_dq_flags"] == 0, "DQ flags must be non-null booleans"
assert post["flag_mismatches"] == 0, "DQ flags do not match their definitions"
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("chartevents:OK")